# Applications of DigiMicPy to Specific Systems 


The general DigiMicPy models can be applied to specific microbial systems, ranging from animal gut microbiomes to aquatic microbial communities. To apply the software to specific hosts or environments, typical modifications may include: 

- Environmental parameters (e.g., temperature range)
- Microbial community structure (e.g., uptake matrix)
- Downstream analysis fit for the specific system (e.g., local stability)

Specifically, temperature-dependence and spatial distribtion are important features of these applications. 

# Example 1: Mice Gut Microbiome 


The mice gut microbiome is mainly composed of Bacteroidetes and Firmicutes. Being a complex real-world system, it is extremely challenging to track the dynamics of each species using the MiCRM or GLV. However, the general microbial models can still be modified and applied to model mice microbial communities, and simulate their responses to perturbations like infection-induced temperature changes. 

Below shows one example of applying general microbial community models to the mice gut microbiome, with a focus on temperature-dependent changes and spatial distribution of gut compartments. 

In [ ]:
# import packages 

import numpy as np
from scipy.stats import multivariate_normal
from scipy.integrate import solve_ivp
from numpy.random import default_rng
import os 

In [ ]:
# temperature-dependent uptake and respiration rates: parameters and functions 

def randtemp_param(N, kw): # generate random temperature-dependent traits for consumer species 
    rng=kw.get('rng',np.random) 

    L = kw['L'] # leakage 
    rho_t = kw['rho_t'] # correlation coefficient for covariance between activation energy and baseline uptake / mortality rates 
    L_v = np.mean(L)
    B0_m = -1.4954 # baseline mortality / respiration rate 
    B0_CUE = 0.1953 # baseline carbon use efficiency parameter 
    B0_u = np.log(np.exp(B0_m) / (1 - L_v - B0_CUE)) # baseline uptake rate 
    B0 = np.array([B0_u, B0_m]) 
    B0_var = 0.17 * np.abs(B0) 
    E_mean = np.array([0.8146, 0.5741]) # mean activation energy for uptake and respiration 
    E_var = 0.1364 * E_mean 
    cov_xy = rho_t * np.sqrt(B0_var * E_var) # covariance between activation energy and baseline uptake / mortality rates

    cov_u = np.array([[B0_var[0], cov_xy[0]], [cov_xy[0], E_var[0]]]) # covariance matrix for uptake
    cov_m = np.array([[B0_var[1], cov_xy[1]], [cov_xy[1], E_var[1]]]) # covariance matrix for respiration

    allu = multivariate_normal.rvs(mean=[B0[0], E_mean[0]], cov=cov_u, size=N).T # draw random samples from multivariate normal distribution for uptake
    allm = multivariate_normal.rvs(mean=[B0[1], E_mean[1]], cov=cov_m, size=N).T # draw random samples from multivariate normal distribution for respiration

    B = np.column_stack((np.exp(allu[0]), np.exp(allm[0]))) # exponentiate the base rates to get the actual values
    E = np.column_stack((allu[1], allm[1])) # activation energy 

    Tpu = 273.15 + rng.normal(35, 5, N) # draw random peak temperatures for uptake from a normal distribution with mean 35 and std 5
    Tpm = Tpu + 3 # peak temperature for respiration is 3 degrees higher than for uptake
    Tp = np.column_stack((Tpu, Tpm)) 

    return B, E, Tp


def temp_trait(B, E, Tp, T, Tr, Ed):
    
    k = 0.0000862 

    # Arrhenius function with high-temp deactivation

    # uptake rate u(T)
    temp_u = B[:, 0] * np.exp((-E[:, 0] / k) * ((1 / T) - (1 / Tr))) / \
              (1 + (E[:, 0] / (Ed - E[:, 0])) * np.exp(Ed / k * (1 / Tp[:, 0] - 1 / T)))

    # respiration rate m(T)
    temp_m = B[:, 1] * np.exp((-E[:, 1] / k) * ((1 / T) - (1 / Tr))) / \
              (1 + (E[:, 1] / (Ed - E[:, 1])) * np.exp(Ed / k * (1 / Tp[:, 1] - 1 / T)))

    tt = np.column_stack((temp_u, temp_m))

    return tt



In [ ]:
# MiCRM functions for later parameter generation 

def F_m(N, M, kw):
    
    if 'tt' in kw:      
        return kw['tt'][:, 1] 
    else:
        return np.full(N, 0.2)


def F_rho(N, M, kw):
    return np.ones(M)


def F_omega(N, M, kw):
    return np.ones(M)

# modular uptake and leakage 

def modular_uptake(N, M, N_modules, s_ratio, rng):
    assert N_modules <= M and N_modules <= N, "N_modules must be less than or equal to both M and N"

    # Baseline calculations
    sR = M // N_modules
    dR = M - (N_modules * sR)

    sC = N // N_modules
    dC = N - (N_modules * sC)

    # Get module sizes for M
    diffR = np.full(N_modules, sR, dtype=int)
    diffR[rng.choice(N_modules, dR, replace=False)] += 1
    mR = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffR) - diffR + 1), np.cumsum(diffR))]

    # Get module sizes for N
    diffC = np.full(N_modules, sC, dtype=int)
    diffC[rng.random.choice(N_modules, dC, replace=False)] += 1
    mC = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffC) - diffC + 1), np.cumsum(diffC))]

    # Preallocate u matrix
    u = rng.random(N, M)

    # Apply scaling
    for x, y in zip(mC, mR):
        u[np.ix_(x, y)] *= s_ratio

    # Normalize each row
    for i in range(N):
        u[i, :] /= np.sum(u[i, :])

    return u


def F_u(N, M, kw): # incorporates modularity 
    rng = kw.get("rng", np.random)

    u_pref = modular_uptake(
        N,
        M,
        kw["N_modules"],
        kw["s_ratio"],
        rng
    )

    if "tt" in kw:
        u_sum = kw["tt"][:,0]
    else:
        u_sum = np.full(N, 2.5)

    return u_pref * u_sum[:,None]


def modular_leakage(M, N_modules, s_ratio, λ, rng):
    assert N_modules <= M, "N_modules must be less than or equal to M"

    # Baseline
    sR = M // N_modules
    dR = M - (N_modules * sR)

    # Get module sizes and add to make to M
    diffR = np.full(N_modules, sR, dtype=int)
    diffR[rng.choice(N_modules, dR, replace=False)] += 1
    mR = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffR) - diffR + 1), np.cumsum(diffR))]

    l = rng.random(M, M)

    for i, x in enumerate(mR):
        for j, y in enumerate(mR):
            if i == j or i + 1 == j:
                l[np.ix_(x, y)] *= s_ratio

    for i in range(M):
        l[i, :] = λ * l[i, :] / np.sum(l[i, :])

    return l


def F_l(N, M, kw): # incorporates modularity 
    rng = kw.get("rng", np.random)

    L = kw["L"]
    l = np.zeros((N, M, M))

    for i in range(N):
        l[i] = modular_leakage(
            M,
            kw["N_modules"],
            kw["s_ratio"],
            L[i],
            rng
        )

    return l

In [ ]:
# generate parameters

def generate_params(N,
                     M,
                     f_m=F_m,
                     f_rho=F_rho,
                     f_omega=F_omega,
                     f_u=F_u,
                     f_l=F_l,
                     **kwargs):


    kw = dict(kwargs)
    B, E, Tp = randtemp_param(N, kw) # Generate species-specific thermal traits

    tt = temp_trait(
            B,
            E,
            Tp,
            kw["T"],
            kw["Tr"],
            kw["Ed"]
        ) # tt = thermal traits; evaluate traits at the current temperature

    kw["tt"] = tt

 
    m = f_m(N, M, kw) 
    u = f_u(N, M, kw) 
    l = f_l(N, M, kw)     


    lambda_ = np.sum(l, axis=2) 

 
    rho = f_rho(N, M, kw)
    omega = f_omega(N, M, kw)


    params = {
        'N': N,
        'M': M,
        'u': u,
        'm': m,
        'l': l,
        'rho': rho,
        'omega': omega,
        'lambda': lambda_,
        'L': kw['L'],
        'B': B,
        'E': E,
        'Tp': Tp,
        'tt': tt
    }
   
    params.update(kwargs)

    return params 


Simulation code for a single temperature (to visualise temporal dynamics of consumers and resources): 

In [ ]:
# simulation code for a single temperature (visualising temporal dynamics)


# set parameters 
N = 10
M = 5
λ = 0.1
N_modules = 2
s_ratio = 10.0
rho_t = 0.5
T = 273.15 + 30
Tr = 273.15 + 10
Ed = 3.5


# MiCRM solver function 
def dCdt_Rdt(t, y, structural):

    N = structural["N"]
    M = structural["M"]

    u = structural["u"]
    m = structural["m"]
    l = structural["l"]
    rho = structural["rho"]
    omega = structural["omega"]
    lambda_alpha = structural["lambda"]

    C = y[:N]
    R = y[N:]

    dCdt = np.zeros(N)
    dRdt = np.zeros(M)

    # Consumer dynamics
    for i in range(N):

        growth = sum(
            C[i] * R[alpha] * u[i, alpha] * (1 - lambda_alpha[i, alpha])
            for alpha in range(M)
        )

        dCdt[i] = growth - C[i] * m[i]

    # Resource dynamics
    for alpha in range(M):

        dRdt[alpha] = rho[alpha] - omega[alpha] * R[alpha]

        # Resource consumption
        dRdt[alpha] -= sum(
            C[i] * R[alpha] * u[i, alpha]
            for i in range(N)
        )

        # Leakage
        dRdt[alpha] += sum(
            sum(
                C[i] * R[beta] * u[i, beta] * l[i, beta, alpha]
                for beta in range(M)
            )
            for i in range(N)
        )

    return np.concatenate((dCdt, dRdt))

# Run one microbial community at a fixed temperature

def run_single_replicate(rep_id=1):

    rng = np.random.default_rng(42 + rep_id)

    # Generate one microbial community
    structural = generate_params(
        N=N,
        M=M,
        L=np.full(N, λ),
        rho_t=rho_t,
        T=T,
        Tr=Tr,
        Ed=Ed,
        N_modules=N_modules,
        s_ratio=s_ratio,
        rng=rng,
    )

    # Initial conditions
    C0 = np.full(N, 0.01)
    R0 = np.full(M, 1.0)
    Y0 = np.concatenate([C0, R0])

    # Time span
    t_span = (0, 50)
    t_eval = np.linspace(*t_span, 300)

    # Run MiCRM
    sol = solve_ivp(
        dCdt_Rdt,
        t_span,
        Y0,
        args=(structural,),
        t_eval=t_eval,
    )

    return {
        "replicate": rep_id,
        "structural": structural,
        "solution": sol,
    }


# ==========================================================
# Run simulation
# ==========================================================

result = run_single_replicate()

print(result["solution"].success)

Simulation code for continuous temperature modelling: 

In [ ]:
# simulation code for continuous temperature modelling 

# function to update temperature

"""
Evaluate an existing microbial community at a new temperature
Structural traits (B, E, Tp, uptake preferences, leakage, etc remain unchanged
Only the temperature-dependent traits are updated 
"""

def update_temperature(structural, T):
    """
    Evaluate an existing microbial community at a new temperature.

    Structural traits (B, E, Tp, uptake preferences, leakage, etc.)
    remain unchanged. Only the temperature-dependent parameters
    (uptake and mortality) are updated.
    """

    tt = temp_trait(
        structural["B"],
        structural["E"],
        structural["Tp"],
        T,
        structural["Tr"],
        structural["Ed"],
    )

    params = structural.copy()

    params["T"] = T
    params["tt"] = tt

    # Update temperature-dependent parameters
    params["m"] = tt[:, 1]
    params["u"] = structural["u_pref"] * tt[:, 0][:, None]

    return params


# output directory
outdir = "output"
os.makedirs(outdir, exist_ok=True)


# model parameters

N = 50
M = 25
L = np.full(N, 0.3)
N_modules = 3
s_ratio = 10
rho_t = 0.5
Tr = 273.15 + 10
Ed = 3.5
temp_vals = np.linspace(273.15 + 10,
                        273.15 + 40,
                        16)
tint = 1000
ttscle = 200
t_eval = np.linspace(0, tint, ttscle)

# set initial conditions 

C0 = np.full(N, 0.1)
R0 = np.full(M, 1.0)
Y0 = np.concatenate([C0, R0])

# run a single replicate 

def run_single_replicate(rep_id=1):

    rng = np.random.default_rng(111 + rep_id)

    # Generate one microbial community
    structural = generate_params(
        N=N,
        M=M,
        L=L,
        rho_t=rho_t,
        T=Tr,
        Tr=Tr,
        Ed=Ed,
        N_modules=N_modules,
        s_ratio=s_ratio,
        rng=rng,
    )

    results = []

    # Evaluate the same microbial community across temperatures
    for T in temp_vals:

        params = update_temperature(structural, T)

        sol = solve_ivp(
            dCdt_Rdt,
            (0, tint),
            Y0,
            args=(params,),
            t_eval=t_eval,
            method="LSODA",
            rtol=1e-4,
            atol=1e-7,
        )

        results.append({
            "temperature": T,
            "solution": sol,
        })

    return {
        "replicate": rep_id,
        "structural": structural,
        "results": results,
    }

# repeat simulation across a number of microbial communities to obtain averages 

simulation = run_single_replicate()

print("Simulation complete.")